# Automated Web Security Scanner

IBM SkillsBuild Cyber Security 2026 student project. This lightweight scanner checks selected HTTP security headers and basic CORS response headers. Only scan systems you own or are authorized to assess.


In [ ]:
import json
from datetime import datetime, timezone
from urllib.parse import urlparse
import requests

SECURITY_HEADERS = {
    'Strict-Transport-Security': 'Review HSTS deployment on HTTPS sites.',
    'Content-Security-Policy': 'Define a policy appropriate to application resources.',
    'X-Content-Type-Options': 'Consider X-Content-Type-Options: nosniff.',
    'X-Frame-Options': 'Consider DENY or SAMEORIGIN, or CSP frame-ancestors.',
    'Referrer-Policy': 'Choose a policy suitable for privacy and application needs.',
    'Permissions-Policy': 'Disable browser features the application does not need.'
}

def audit_url(url, timeout=8):
    url = url.strip()
    if '://' not in url:
        url = 'https://' + url
    parsed = urlparse(url)
    if parsed.scheme not in ('http', 'https') or not parsed.hostname:
        raise ValueError('Enter a valid http:// or https:// URL.')
    response = requests.get(url, timeout=timeout, allow_redirects=True,
        headers={'User-Agent': 'IBM-SkillsBuild-WebSecurityScanner/1.0'})
    headers = response.headers
    findings = []
    for name, recommendation in SECURITY_HEADERS.items():
        present = name in headers
        findings.append({'category':'security_header','name':name,'present':present,
            'observed_value':headers.get(name),'severity':'info' if present else 'medium',
            'recommendation':recommendation})
    origin = headers.get('Access-Control-Allow-Origin')
    credentials = headers.get('Access-Control-Allow-Credentials', '').lower()
    if origin == '*' and credentials == 'true':
        cors_note = 'Wildcard origin and credentials indicator; review CORS configuration. Browsers reject this exact combination for credentialed reads.'
        severity = 'high'
    elif origin == '*':
        cors_note = 'Wildcard origin observed; may be appropriate for public non-credentialed resources.'
        severity = 'low'
    elif origin:
        cors_note = f'Explicit allowed origin observed: {origin}. Confirm it is intentional.'
        severity = 'info'
    else:
        cors_note = 'No Access-Control-Allow-Origin header observed; this is not automatically a vulnerability.'
        severity = 'info'
    findings.append({'category':'cors','name':'CORS configuration observation','severity':severity,
        'observation':cors_note,'allow_origin':origin,'allow_credentials':credentials})
    findings.append({'category':'transport','name':'HTTPS','severity':'info' if urlparse(response.url).scheme == 'https' else 'high',
        'observation':f'Final URL: {response.url}'})
    return {'target':url,'checked_at_utc':datetime.now(timezone.utc).isoformat(),
        'http_status':response.status_code,'final_url':response.url,'findings':findings}

# Change this to a website you own or are authorized to test.
TARGET_URL = 'https://example.com'
result = audit_url(TARGET_URL)
print(json.dumps(result, indent=2))
with open('scan_report.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2)
print('Saved scan_report.json')


## Interpretation and limitations
Missing headers are not automatic proof of a vulnerability. CORS behavior depends on request origin, route, and credentials. This notebook performs a single non-destructive GET request and is not a full penetration test. Validate findings manually.
